# Lab 3: Backpropagation, by hand

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Sep 23, 11:59 PM ET · **Target time:** about 75–90 minutes

---

## What this lab is for

Lab 2 ended on a cliffhanger. You trained a 49-parameter network with finite
differences, it worked, and it cost **20,001 forward passes** to do it. The
closing line was that autograd computes the same gradient without paying that
price.

This lab is you building the thing that makes that true. By the end you will
have written a working automatic differentiation engine, in about forty lines,
and used it to get the exact same gradient Lab 2 approximated.

Three parts, each one a step up in size:

1. Backprop by hand on the small graph from lecture, so the bookkeeping is
   concrete before any code exists.
2. A `Value` class that does it automatically for any graph you can build.
3. The matrix version, on Lab 2's network, with the forward-pass count as the
   punchline.

No PyTorch. Next week you will meet `loss.backward()` and it will be doing
exactly what you wrote here.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **You may work together.** Everyone runs the same seed, so your numbers
  *should* match your neighbor's.
- **The understanding is assessed on the quiz after the due date**, which is
  Quiz 4. Part of that quiz asks you to explain **what this lab demonstrated and
  why**. Quiz 3 does not depend on this lab, because the lab is now due after it.
- **Solutions are posted after the deadline.**

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end. An
agent can write this code in a minute. It cannot sit Thursday's quiz for you.

## Step 0: Setup

Same `6600` seed as lecture and Lab 2. The numbers you print should match the
ones on the slides, and the network in Part 3 is the identical network you
trained last week.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 6600
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("ready")

---

# Part 1 · Backprop by hand

The graph from lecture. Three inputs, four operations:

$$C = A \times B \qquad F = D + C \qquad G = F + A \qquad H = G \times B$$

with $A = 2$, $B = 3$, $D = 5$.

**Notice that $A$ and $B$ are each used twice.** $A$ feeds both $C$ and $G$;
$B$ feeds both $C$ and $H$. That is the whole reason backprop needs the rule
about summing over paths, and it is the thing most people get wrong the first
time.

Run the forward pass first.

In [ ]:
A, B, D = 2.0, 3.0, 5.0

C = A * B
F = D + C
G = F + A
H = G * B

for name, v in [("C = A*B", C), ("F = D+C", F), ("G = F+A", G), ("H = G*B", H)]:
    print(f"{name:12s} = {v:5.1f}")

## 1a · The backward pass, on paper

Work backwards from $H$, one node at a time. Start with
$\partial H / \partial H = 1$ and apply the three rules from lecture:

- **Multiply node** $u = v \times w$: each input gets the upstream gradient
  times *the other* input.
- **Add node** $u = v + w$: each input gets the upstream gradient unchanged.
- **A node used more than once**: its gradient is the **sum** of the gradients
  arriving along each path.

Fill in the numbers below. Do it by hand before you run anything; the next cell
checks you.

In [ ]:
# TODO: fill in each partial derivative of H with respect to the node.
#   Work backwards: H, then G and B's first contribution, then F, then C and D,
#   then A. Remember that A and B each collect from two paths.
dH_dH = 1.0
dH_dG = None    # H = G*B, so this is the other input to that multiply
dH_dF = None    # G = F+A, an add
dH_dD = None    # F = D+C, an add
dH_dC = None    # F = D+C, an add
dH_dA = None    # two paths: through C, and through G
dH_dB = None    # two paths: through C, and through H

if None in (dH_dG, dH_dF, dH_dD, dH_dC, dH_dA, dH_dB):
    raise NotImplementedError("Part 1a: fill in the six partials")

## 1b · Check yourself numerically

You do not need to trust the hand arithmetic. Lab 2's finite-difference trick
works on any function, including this one: nudge an input, see how much $H$
moves.

$$\frac{\partial H}{\partial A} \approx \frac{H(A + \varepsilon) - H(A - \varepsilon)}{2\varepsilon}$$

This is the **central** difference, which is more accurate than the one-sided
version Lab 2 used. More on that in Part 3.

In [ ]:
def evaluate(a, b, d):
    """Recompute H from scratch for any inputs."""
    c = a * b
    f = d + c
    g = f + a
    return g * b


def numeric_grad(f, a, b, d, eps=1e-6):
    """Central differences with respect to each of the three inputs."""
    return (
        (f(a + eps, b, d) - f(a - eps, b, d)) / (2 * eps),
        (f(a, b + eps, d) - f(a, b - eps, d)) / (2 * eps),
        (f(a, b, d + eps) - f(a, b, d - eps)) / (2 * eps),
    )


num_A, num_B, num_D = numeric_grad(evaluate, A, B, D)

print(f"{'':6s} {'by hand':>10s} {'numeric':>10s}   match")
for name, hand, num in [("dH/dA", dH_dA, num_A),
                        ("dH/dB", dH_dB, num_B),
                        ("dH/dD", dH_dD, num_D)]:
    ok = "yes" if abs(hand - num) < 1e-4 else "NO"
    print(f"{name:6s} {hand:10.4f} {num:10.4f}   {ok}")

**If `dH/dA` came out as 9 or as 3, you took one path and stopped.** The answer
is 12, because $A$ reaches $H$ two different ways and both contributions count.
That single fact is what the `+=` in Part 2 is there to enforce.

---

# Part 2 · An autodiff engine in forty lines

Doing that by hand does not scale past about six nodes. So make the graph build
itself.

The idea: a `Value` holds a number, and every operation on `Value`s returns a
new `Value` that **remembers what made it**. Once the forward pass has run you
have a graph, and the backward pass is a walk through that graph in reverse.

Each node stores four things:

| Field | Holds |
|---|---|
| `.data` | the number from the forward pass |
| `.grad` | the derivative of the output with respect to this node |
| `._prev` | the nodes that fed this one |
| `._backward` | a function that pushes this node's grad to `._prev` |

`_backward` is the only clever part, and it is just the chain rule for one
operation, written as a closure so it can be called later.

Read `__add__` carefully. You are writing the other two.

In [ ]:
class Value:
    """A scalar that remembers how it was computed."""

    def __init__(self, data, children=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None    # leaves have nothing to push back

    def __repr__(self):
        return f"Value({self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        # WORKED EXAMPLE. Study this one, then write the next two.
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # d(out)/d(self) = 1 and d(out)/d(other) = 1, so an add just
            # passes the upstream gradient through to both inputs.
            # It is += and not =, because either input may also be feeding
            # some other node. That is Part 1's "sum over paths" rule.
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # TODO: build `out` the same way __add__ does, with data = the product,
        #   children = (self, other), and op = "*". Then write _backward:
        #   d(out)/d(self) is other.data, and d(out)/d(other) is self.data.
        #   Use += for both.
        raise NotImplementedError("Value.__mul__")

    def tanh(self):
        t = np.tanh(self.data)
        # TODO: return a Value holding t, with children = (self,) and op = "tanh".
        #   The derivative of tanh is 1 - tanh(x)**2, and you already have
        #   tanh(x) sitting in `t`, so there is no need to recompute it.
        raise NotImplementedError("Value.tanh")

    # so that 2 * v and v + 2 work as well as v * 2 and 2 + v
    __rmul__ = __mul__
    __radd__ = __add__

## 2a · Ordering the backward pass

One constraint makes this work: **you cannot compute a node's gradient until
every node it feeds already has its own.** If $A$ feeds both $C$ and $G$, then
$C$ and $G$ must both be finished before $A$ is touched, or $A$ will collect
half its gradient and be wrong.

That ordering is a **topological sort**, the Week 3 idea. Sort the nodes so
every node comes after its inputs, then walk the list backwards.

This one is given to you.

In [ ]:
def topological_order(root):
    """Return nodes so that every node appears after the nodes that feed it.

    Depth-first: a node is appended only once all of its children have been,
    which is exactly the property the backward pass needs, reversed.
    """
    order, seen = [], set()

    def visit(node):
        if id(node) in seen:
            return
        seen.add(id(node))
        for child in node._prev:
            visit(child)
        order.append(node)

    visit(root)
    return order


print("topological_order defined")

## 2b · The backward pass

Now the eight lines that are the whole point of the week.

Three steps:

1. Get the topological order of everything feeding the output.
2. Seed the output's gradient at **1.0**. The derivative of the output with
   respect to itself is 1, and every other gradient is built from that.
3. Walk the list **in reverse**, calling each node's `_backward()`.

Also zero the gradients first. Gradients accumulate with `+=`, which is what
makes the multi-path rule work, but it also means a second `backward()` call on
a dirty graph would double everything.

In [ ]:
def backward(root):
    """Fill in .grad on every node feeding `root`."""
    order = topological_order(root)

    # TODO: three steps.
    #   1. set .grad = 0.0 on every node in `order`, so repeat calls are safe
    #   2. set root.grad = 1.0
    #   3. for node in reversed(order): node._backward()
    raise NotImplementedError("backward")

## 2c · Does it agree with Part 1?

Same graph, same three inputs. You did this by hand twenty minutes ago; now the
machine should get the same three numbers without being told anything about the
structure.

In [ ]:
vA, vB, vD = Value(2.0), Value(3.0), Value(5.0)

vC = vA * vB
vF = vD + vC
vG = vF + vA
vH = vG * vB

backward(vH)

print(f"H = {vH.data:.1f}   (Part 1 got {H:.1f})")
print()
print(f"{'':7s} {'engine':>8s} {'by hand':>9s}   match")
for name, node, hand in [("dH/dA", vA, dH_dA),
                         ("dH/dB", vB, dH_dB),
                         ("dH/dD", vD, dH_dD)]:
    ok = "yes" if abs(node.grad - hand) < 1e-9 else "NO"
    print(f"{name:7s} {node.grad:8.4f} {hand:9.4f}   {ok}")

## 2d · A graph you would not do by hand

The engine does not care how big the graph is. Here is a 2-input, 3-hidden-unit,
1-output tanh network written one scalar at a time, with a squared-error loss.
Seventeen parameters, every one of them differentiated by the same eight lines.

Nobody would write a network this way. The point is that you no longer have to
derive anything: build the forward pass, call `backward`, read the gradients.

In [ ]:
rng = np.random.default_rng(SEED)

# 2 -> 3 -> 1, scalar by scalar. W is (units, inputs), as always.
# The g prefix is just to keep these clear of the arrays in Part 3.
gW1 = [[Value(rng.normal(0, 0.8)) for _ in range(2)] for _ in range(3)]
gb1 = [Value(0.0) for _ in range(3)]
gW2 = [Value(rng.normal(0, 0.8)) for _ in range(3)]
gb2 = Value(0.0)

params = [w for row in gW1 for w in row] + gb1 + gW2 + [gb2]

x_in = [Value(0.5), Value(-1.2)]
y_target = 1.0

h = [(gW1[j][0] * x_in[0] + gW1[j][1] * x_in[1] + gb1[j]).tanh() for j in range(3)]
pred = gW2[0] * h[0] + gW2[1] * h[1] + gW2[2] * h[2] + gb2
diff = pred + (-1.0) * y_target
loss_node = diff * diff

backward(loss_node)

print(f"parameters : {len(params)}")
print(f"nodes in graph: {len(topological_order(loss_node))}")
print(f"prediction : {pred.data:.4f}   target: {y_target}")
print(f"loss       : {loss_node.data:.4f}")
print()
print("a few gradients:")
for label, p in [("dL/dgW1[0][0]", gW1[0][0]), ("dL/dgb1[0]", gb1[0]),
                 ("dL/dgW2[0]", gW2[0]), ("dL/dgb2", gb2)]:
    print(f"  {label:14s} = {p.grad:+.6f}")

Sanity-check one of those against finite differences. Nudge a single weight,
rebuild the whole graph, see how the loss moved.

In [ ]:
def loss_with(w00):
    """Recompute the loss with gW1[0][0] set to w00, everything else fixed."""
    hh = []
    for j in range(3):
        w0 = w00 if j == 0 else gW1[j][0].data
        hh.append(np.tanh(w0 * x_in[0].data + gW1[j][1].data * x_in[1].data
                          + gb1[j].data))
    p = sum(gW2[k].data * hh[k] for k in range(3)) + gb2.data
    return (p - y_target) ** 2


eps = 1e-6
w = gW1[0][0].data
numeric = (loss_with(w + eps) - loss_with(w - eps)) / (2 * eps)

print(f"engine  : {gW1[0][0].grad:+.8f}")
print(f"numeric : {numeric:+.8f}")
print(f"abs diff: {abs(gW1[0][0].grad - numeric):.2e}")

---

# Part 3 · The matrix version, on Lab 2's network

Scalar `Value` objects are a fine way to *understand* backprop and a terrible
way to *run* it. Real frameworks do the same thing on whole arrays at once.

This is Lab 2's network, exactly: $1 \to 16 \to 1$ with tanh, 49 parameters,
fitting a noisy sine, same seed. Everything below the loss is provided. You are
writing the backward pass.

From lecture, a layer computing $\mathbf{z} = \mathbf{W}\mathbf{x} +
\mathbf{b}$ owes three things:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}}\, \mathbf{x}^{\top},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{b}}
  = \frac{\partial \mathcal{L}}{\partial \mathbf{z}},
  \qquad
  \frac{\partial \mathcal{L}}{\partial \mathbf{x}}
  = \mathbf{W}^{\top} \frac{\partial \mathcal{L}}{\partial \mathbf{z}}$$

The first two get updated. The third is what the previous layer receives.

**If you get stuck, check shapes.** As lecture put it: only one arrangement of
$\mathbf{W}$, $\mathbf{W}^{\top}$ and the incoming gradient produces each
required shape, so if the shapes work the formula is almost certainly right.

In [ ]:
HIDDEN, STEPS, LR, EPS = 16, 400, 0.1, 1e-5
SHAPES = [("W", (HIDDEN, 1)), ("b", (HIDDEN,)), ("W", (1, HIDDEN)), ("b", (1,))]
N_PARAMS = sum(int(np.prod(s)) for _, s in SHAPES)

rng_data, rng_init = np.random.default_rng(SEED), np.random.default_rng(SEED)
x = np.linspace(-1, 1, 200).reshape(-1, 1)
y_true = np.sin(3.0 * x.ravel())
y = y_true + rng_data.normal(0, 0.10, 200)
N = x.shape[0]


def unpack(theta):
    out, i = [], 0
    for _, shape in SHAPES:
        n = int(np.prod(shape))
        out.append(theta[i:i + n].reshape(shape))
        i += n
    return out


calls = 0


def forward(X, theta):
    global calls
    calls += 1
    W1, b1, W2, b2 = unpack(theta)
    return (np.tanh(X @ W1.T + b1) @ W2.T + b2).ravel()


def loss(theta):
    return float(np.mean((forward(x, theta) - y) ** 2))


theta0 = np.zeros(N_PARAMS)
i = 0
for kind, shape in SHAPES:
    n = int(np.prod(shape))
    theta0[i:i + n] = (rng_init.normal(0, 1 / np.sqrt(shape[1]), n)
                       if kind == "W" else 0.0)
    i += n

print(f"parameters: {N_PARAMS}   (same 49 as Lab 2)")
print(f"data: {N} points, noise sd 0.10")
print(f"initial loss: {loss(theta0):.6f}")
calls = 0

## 3a · The backward pass

The loss is $\mathcal{L} = \frac{1}{N}\sum_i (\hat{y}_i - y_i)^2$, so the
gradient entering the network is

$$\frac{\partial \mathcal{L}}{\partial \hat{y}} = \frac{2(\hat{y} - y)}{N}$$

From there it is the three-line recipe above, twice, with the tanh derivative
in between. Because this is a batch of 200 examples rather than one, the outer
products become matrix products and the bias gradients become **sums down the
batch axis**.

Shapes to aim for, so you can check yourself as you go:

| Quantity | Shape |
|---|---|
| `dz2` | (200, 1) |
| `dW2` | (1, 16) |
| `db2` | (1,) |
| `da1` | (200, 16) |
| `dz1` | (200, 16) |
| `dW1` | (16, 1) |
| `db1` | (16,) |

In [ ]:
def backprop_gradient(theta):
    """Return (grad, loss) in a single forward and a single backward pass."""
    W1, b1, W2, b2 = unpack(theta)

    # forward, keeping every activation because the backward pass needs them
    z1 = x @ W1.T + b1          # (200, 16)
    a1 = np.tanh(z1)            # (200, 16)
    z2 = a1 @ W2.T + b2         # (200, 1)
    pred = z2.ravel()           # (200,)
    L = float(np.mean((pred - y) ** 2))

    # TODO: the backward pass, five steps.
    #   dz2  = (2.0 * (pred - y) / N).reshape(-1, 1)
    #   dW2  = dz2.T @ a1              and    db2 = dz2.sum(axis=0)
    #   da1  = dz2 @ W2                (gradient handed back to layer 1)
    #   dz1  = da1 * (1.0 - a1 ** 2)   (the tanh derivative)
    #   dW1  = dz1.T @ x               and    db1 = dz1.sum(axis=0)
    dW1 = db1 = dW2 = db2 = None
    if dW1 is None:
        raise NotImplementedError("backprop_gradient")

    grad = np.concatenate([dW1.ravel(), db1.ravel(), dW2.ravel(), db2.ravel()])
    return grad, L

## 3b · Gradient check against Lab 2

Here is Lab 2's finite-difference gradient, unchanged, and a central-difference
version alongside it. Compare all 49 numbers.

In [ ]:
def finite_difference_gradient(theta, eps=EPS):
    """Lab 2's one-sided version: one extra forward pass per parameter."""
    base = loss(theta)
    grad = np.zeros_like(theta)
    for j in range(len(theta)):
        bumped = theta.copy()
        bumped[j] += eps
        grad[j] = (loss(bumped) - base) / eps
    return grad, base


def central_difference_gradient(theta, eps=1e-6):
    """Two extra forward passes per parameter, and far more accurate."""
    grad = np.zeros_like(theta)
    for j in range(len(theta)):
        up, down = theta.copy(), theta.copy()
        up[j] += eps
        down[j] -= eps
        grad[j] = (loss(up) - loss(down)) / (2 * eps)
    return grad


g_back, L_back = backprop_gradient(theta0)
g_fd, L_fd = finite_difference_gradient(theta0)
g_cd = central_difference_gradient(theta0)

print(f"loss agrees: {abs(L_back - L_fd):.2e}")
print()
print(f"backprop vs one-sided differences  max abs diff: {np.abs(g_back - g_fd).max():.2e}")
print(f"backprop vs central differences    max abs diff: {np.abs(g_back - g_cd).max():.2e}")
print()
print(f"{'param':>6s} {'backprop':>12s} {'one-sided':>12s} {'central':>12s}")
for j in list(range(4)) + [N_PARAMS - 1]:
    print(f"{j:6d} {g_back[j]:12.6f} {g_fd[j]:12.6f} {g_cd[j]:12.6f}")

**Read the two max-diff numbers.** Backprop and central differences agree to
about 1e-10. Backprop and Lab 2's one-sided differences only agree to about
1e-5, five orders of magnitude worse.

Backprop did not get less accurate between those two lines. **Backprop is exact**,
up to floating-point rounding; it evaluates the derivative rather than
estimating it. The disagreement is entirely the finite-difference
approximation, whose one-sided error shrinks like $\varepsilon$ while the
central version's shrinks like $\varepsilon^2$.

So Lab 2 was paying 20,001 forward passes for a gradient that was *also* the
least accurate of the three.

---

# Part 4 · What it cost

The last thing to establish is that this was worth doing. Train the same
network for the same 400 steps with the same learning rate, once each way, and
count forward passes.

Lab 2 predicted the finite-difference count: one pass for the current loss plus
one per parameter, every step, so $400 \times (1 + 49) + 1 = 20{,}001$.

Predict the backprop count before you run it.

In [ ]:
theta = theta0.copy()
calls = 0
hist_bp = []

# TODO: STEPS of gradient descent, using backprop_gradient this time.
#   grad, L = backprop_gradient(theta)
#   hist_bp.append(L)
#   theta = theta - LR * grad
# Note that backprop_gradient does its own forward pass inline and never calls
# forward(), so bump `calls` by one yourself each step to keep the count honest.
if len(hist_bp) < STEPS:
    raise NotImplementedError("training loop: take STEPS steps with backprop")

hist_bp.append(loss(theta))
theta_bp = theta
calls_bp = calls

print(f"start loss : {hist_bp[0]:.6f}")
print(f"end loss   : {hist_bp[-1]:.6f}")
print(f"forward passes: {calls_bp:,}")

Now the same run with finite differences, for the comparison. This one is
provided because you already wrote it in Lab 2. It is also timed, so that the
saving shows up as something other than a pass count.

In [ ]:
import time

theta = theta0.copy()
calls = 0
hist_fd = []

t0 = time.perf_counter()
for _ in range(STEPS):
    grad, base = finite_difference_gradient(theta)
    hist_fd.append(base)
    theta = theta - LR * grad
hist_fd.append(loss(theta))
elapsed_fd = time.perf_counter() - t0
calls_fd = calls
theta_fd = theta

t0 = time.perf_counter()
for _ in range(STEPS):
    backprop_gradient(theta0)
elapsed_bp = time.perf_counter() - t0

print(f"finite differences : {calls_fd:>7,} forward passes   {elapsed_fd:6.2f} s")
print(f"backpropagation    : {calls_bp:>7,} forward passes   {elapsed_bp:6.2f} s")
print()
print(f"passes saved  : {calls_fd / calls_bp:5.1f}x")
print(f"wall clock    : {elapsed_fd / elapsed_bp:5.1f}x")
print()
print(f"final loss, finite differences : {hist_fd[-1]:.6f}")
print(f"final loss, backpropagation    : {hist_bp[-1]:.6f}")

## 4a · Same fit, same curve

Two things to look at: the loss curves should sit on top of each other, and the
two fitted networks should draw the same function.

In both panels the backprop result is the **thick pale line** and the
finite-difference result is the **thin dashed line drawn on top of it**. If you
can only see one curve, that is the answer: the dashes are tracking the thick
line exactly. Nothing was given up.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Backprop goes down thick and pale; finite differences goes on top as a thin
# dashed line. Any disagreement would show up as the dashes wandering off.
ax1.plot(hist_bp, lw=4, alpha=0.4, color="C2",
         label=f"backpropagation ({calls_bp:,} passes)")
ax1.plot(hist_fd, lw=1.4, ls="--", color="C3",
         label=f"finite differences ({calls_fd:,} passes)")
ax1.set_xlabel("step")
ax1.set_ylabel("MSE")
ax1.set_yscale("log")
ax1.set_title("Same trajectory, very different cost")
ax1.legend()

order = np.argsort(x.ravel())
ax2.scatter(x.ravel(), y, s=8, alpha=0.25, color="C0", label="data")
ax2.plot(x.ravel()[order], y_true[order], lw=1.5, color="0.35",
         label="true sin(3x)")
ax2.plot(x.ravel()[order], forward(x, theta_bp)[order], lw=4, alpha=0.4,
         color="C2", label="backpropagation")
ax2.plot(x.ravel()[order], forward(x, theta_fd)[order], lw=1.4, ls="--",
         color="C3", label="finite differences")
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_title("Same fit")
ax2.legend(fontsize=8, loc="lower right")

plt.tight_layout()
plt.show()

print(f"max difference between the two fitted curves: "
      f"{np.abs(forward(x, theta_fd) - forward(x, theta_bp)).max():.2e}")
print(f"max difference between the two loss curves  : "
      f"{np.abs(np.array(hist_fd) - np.array(hist_bp)).max():.2e}")

## 4b · Where this goes

49 parameters is nothing. The gap is a **ratio**, and it grows with the model.

The finite-difference cost is one forward pass per parameter per step. The
backprop cost is one forward and one backward per step, and the backward pass
costs about what the forward pass costs, **no matter how many parameters there
are**. So the ratio is roughly the parameter count itself.

In [ ]:
print(f"{'model':<28s} {'params':>14s} {'FD passes/step':>16s} {'backprop':>10s}")
for name, p in [("this lab", N_PARAMS),
                ("Lab 2's 2-3-2-1", 20),
                ("small CNN", 1_200_000),
                ("ResNet-50", 25_600_000),
                ("GPT-3", 175_000_000_000)]:
    print(f"{name:<28s} {p:>14,} {p + 1:>16,} {'~2':>10s}")

print()
print("The right-hand column does not move. That is the whole reason")
print("deep learning is computationally possible.")

---

# Part 5 · What this lab showed

**This is the part that gets assessed**, on Quiz 4 rather than Quiz 3, since this
lab is now due after Quiz 3. Nothing to memorize and no numbers to transcribe: the
quiz asks *why*, not *what was your value*.

Scroll back through your own output and make sure you can say, in a sentence each:

1. **Why $\partial H / \partial A$ was 12 and not 9 or 3.** What it means for
   a node to be used more than once, and which line of the `Value` class
   enforces it.
2. **What the four fields on a `Value` are for**, and why `_backward` has to be
   stored at forward time rather than derived later.
3. **Why the backward pass needs a topological order.** What goes wrong if you
   process a node before everything it feeds is done.
4. **Why the output's gradient is seeded at 1.0.**
5. **Why the backward pass needs the activations from the forward pass**, and
   what that costs in memory.
6. **Why backprop matched central differences to 1e-9 but Lab 2's one-sided
   differences only to 1e-5.** Which of the three is the approximation.
7. **Why the cost ratio grows with the parameter count.** Why the backward pass
   costs about one forward pass regardless of model size, and what that means
   for a model with 175 billion parameters.

If you can answer those seven, you are ready for the quiz.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Sep 23, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. Part 5 is
your study guide for the quiz that assesses the understanding.

::: {.callout-note}
## Extra week
Week 3 spent most of its time reinforcing fundamentals and only started
backpropagation, so this lab moved from Sep 16 to **Sep 23**. Parts 2 and 3 run
slightly ahead of where lecture stopped; Week 4 covers the rest of the backward
pass before the new due date.
:::